In [1]:
import os, sys
import numpy as np
import pandas as pd
import pickle as pk
import seaborn as sns
from random import randint
from itertools import *
import scipy.interpolate as itp
from multiprocessing import Pool, Process

In [2]:
### cb molecules
def getcbMol(nAZ=2, k='H'):
    # 0 Ca bound states
    cb0 = [f'cb{k}0']

    # 1 Ca bound states
    cb1 = []
    for i in range(nAZ+1):
        cb1.append(f'cb{i}{k}1')

    # 2 Ca bound states
    cb2 = []
    for i in range(nAZ+1):
        for j in range(i,nAZ+1):
            cb2.append(f'cb{i}_{j}{k}2')

    return cb0, cb1, cb2

### cb reactions
def getcbRxn(nAZ=2, k='H'):
    rxns = []
    # 0 to 1 Ca bound states
    for i in range(nAZ+1):
        rxns.append(f"cb{k}0 + Ca{i} <-> cb{i}{k}1      [>2*cb{k}_on, <cb{k}_off]")

    # 1 to 2 Ca bound states
    for i in range(nAZ+1):
        for j in range(nAZ+1):
            if i<=j:
                rxns.append(f"cb{i}{k}1 + Ca{j} <-> cb{i}_{j}{k}2\t\t[>cb{k}_on, <2*cb{k}_off]")
            else:
                rxns.append(f"cb{i}{k}1 + Ca{j} <-> cb{j}_{i}{k}2\t\t[>cb{k}_on, <2*cb{k}_off]")

    return rxns

In [3]:
for nAZ in range(7,36):
    text = '''//------- Calbindin Rates and Parameters ----------

D_cb = 0 //0.28e-6 // Schmidt et al. 2003, J. Physiol.
cb_conc = 40e-6  // since medium and high are modelled as separate mols

// CalB reaction rates
cbH_on  = 0.55e7
cbH_off = 2.6
cbM_on  = 4.35e7
cbM_off = 35.8

// Calbindin eq fractions at [cbH0M0]o = 40 uM and [Ca2]i = ~100 nm
cbM0_feq = 0.794640
cbM1_feq = 0.193611
cbM2_feq = 0.011749
cbH0_feq = 0.681430
cbH1_feq = 0.288101
cbH2_feq = 0.030469

DEFINE_MOLECULES 
{
'''
    
    for k in ['M', 'H']:
        cb0, cb1, cb2 = getcbMol(nAZ=nAZ, k=k)

        for cb in cb0: text += f"\t{cb}\t{{D_3D = D_cb CUSTOM_SPACE_STEP = 0.100 TARGET_ONLY}}\n"
        for cb in cb1: text += f"\t{cb}\t{{D_3D = D_cb CUSTOM_SPACE_STEP = 0.100 TARGET_ONLY}}\n"
        for cb in cb2: text += f"\t{cb}\t{{D_3D = D_cb CUSTOM_SPACE_STEP = 0.100 TARGET_ONLY}}\n"
        text += '\n'


    text += '''}\n\nDEFINE_REACTIONS\n{\n'''

    for k in ['M', 'H']:
        rxns = getcbRxn(nAZ=nAZ, k=k)

        for r in rxns:
            text += '\t' + r + '\n'
        text += '\n'

    text += '''}

INSTANTIATE CB OBJECT
{
    presynaptic_cbM0 RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cbM0
        CONCENTRATION = cbM0_feq*cb_conc
    }
    presynaptic_cb0M1 RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cb0M1
        CONCENTRATION = cbM1_feq*cb_conc
    }
    presynaptic_cb0_0M2 RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cb0_0M2
        CONCENTRATION = cbM2_feq*cb_conc
    }
    presynaptic_cbH0 RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cbH0
        CONCENTRATION = cbH0_feq*cb_conc
    }
    presynaptic_cb0H1 RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cb0H1
        CONCENTRATION = cbH1_feq*cb_conc
    }
    presynaptic_cb0_0H2 RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cb0_0H2
        CONCENTRATION = cbH2_feq*cb_conc
    }
}'''
    with open(f'buffers_AZ_{nAZ}.mdl', 'w+') as of: of.write(text)
    #print(text)

### Calculate calbindin state concentrations

In [ ]:
cbH0M0_feq = 0.54141
cbH0M1_feq = 0.13201
cbH1M0_feq = 0.22901
cbH1M1_feq = 0.05571
cbH0M2_feq = 0.00801
cbH2M0_feq = 0.02422
cbH1M2_feq = 0.003381
cbH2M1_feq = 0.005891
cbH2M2_feq = 0.000358

#print(cbH0M0_feq+cbH0M1_feq+cbH1M0_feq+cbH1M1_feq+cbH0M2_feq+
#      cbH2M0_feq+cbH1M2_feq+cbH2M1_feq+cbH2M2_feq)

cbM0 = (cbH0M0_feq + cbH1M0_feq + cbH2M0_feq)
cbM1 = (cbH0M1_feq + cbH1M1_feq + cbH2M1_feq)
cbM2 = (cbH0M2_feq + cbH1M2_feq + cbH2M2_feq)

cbH0 = (cbH0M0_feq + cbH0M1_feq + cbH0M2_feq)
cbH1 = (cbH1M0_feq + cbH1M1_feq + cbH1M2_feq)
cbH2 = (cbH2M0_feq + cbH2M1_feq + cbH2M2_feq)

# cbM0+cbM1+cbM2+cbH0+cbH1+cbH2
fcb = ['cbM0', 'cbM1', 'cbM2', 'cbH0', 'cbH1', 'cbH2']
fcb = dict(zip(fcb, [cbM0, cbM1, cbM2, cbH0, cbH1, cbH2]))

for k,v in fcb.items():
    print(f'{k}_feq = {v},')
    
fcb = {
    'cbM0_feq': 0.794640,
    'cbM1_feq': 0.193611,
    'cbM2_feq': 0.011749,
    'cbH0_feq': 0.681430,
    'cbH1_feq': 0.288101,
    'cbH2_feq': 0.030469
}
print(sum(fcb.values()))

### Instance

In [ ]:
cb0, cb1, cb2 = getcbMol(nAZ=2, k='H')
#print(cb0, cb1, cb2)
stext = '''
    presynaptic_cb RELEASE_SITE {
        SHAPE = (MFB.bouton[ALL])
        MOLECULE = cbH0
        CONCENTRATION = cbH2M2_feq*cb_conc
    }'''
nca = 1
text = ''
for nca in range(3):
    for cb in eval(f'cb{nca}'):
        text += stext.replace('_cb', f'_{cb}').replace('cbH2M2', f'{cb}')
        if nca:
            nMol = len(eval(f'cb{nca}'))
            text = text.replace(f'{cb}_feq*cb_conc', f'{cb}_feq*conc/{nMol}')

print(text)